In [18]:
import hail as hl
#import argparse
#import pandas
import os

from ukb_utils import hail_init
from ukb_utils import genotypes
from ko_utils import qc
from ko_utils import analysis

In [2]:
import hail as hl
from gnomad.utils.vep import process_consequences

In [3]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('/logs/hail/hail_test_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe018.hpc.in.bmrc.ox.ac.uk:4041
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.74-0c3a74d12093
LOGGING: writing to /logs/hail/hail_test_export.log


In [4]:
mt1 = qc.get_table('data/tmp/ukb_wes_200k_phased_tmp_chr22.1of1.vcf.gz', 'vcf')
#mt1 = qc.get_table('data/tmp/ukb_wes_200k_phased_tmp_chr22.1of1.vcf.gz', 'vcf')
#mt1 = qc.recalc_info(mt1)
mt1 = qc.translate_sample_ids(mt1, from_app='12788', to_app='11867') # translate to lindgren app ID
#mt1 = qc.filter_to_european(mt1)
mt1 = qc.filter_min_missing(mt1, 0.05)
#mt1 = qc.filter_max_maf(mt1, float(0.02))
mt1 = analysis.annotate_vep(mt1, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')
mt1 = analysis.filter_vep(mt1, 'consequence_category',['damaging_missense','ptv'])

Mapping IDs from UKB app 12788 to 11867


2021-09-20 15:37:53 Hail: INFO: Reading table without type imputation
  Loading field 'eid_11867' as type str (not specified)
  Loading field 'eid_12788' as type str (not specified)
2021-09-20 15:37:55 Hail: INFO: Coerced almost-sorted dataset


Annotating with VEP file: data/vep/full/ukb_wes_200k_full_vep_chr22.vcf


In [5]:
mt2 = qc.get_table('data/unphased/tmp/ukb_wes_200k_filtered_2samples_chr22.vcf.bgz', 'vcf')
mt2 = qc.filter_max_mac(mt2, 1) # get singletons
mt2 = qc.filter_min_missing(mt2, 0.05)
mt2 = analysis.annotate_vep(mt2, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')
mt2 = analysis.filter_vep(mt2, 'consequence_category',['damaging_missense','ptv'])

Annotating with VEP file: data/vep/full/ukb_wes_200k_full_vep_chr22.vcf


In [11]:
ko_prob = analysis.gene_csqs_calc_pKO(mt1, mt2)
ko_prob = ko_prob.filter_entries(ko_prob.pKO == 1)
ko_prob.show()
#ko_prob.show()

2021-09-20 15:40:10 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 15:40:10 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 15:40:13 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 15:40:17 Hail: INFO: Coerced sorted dataset
2021-09-20 15:40:29 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 15:40:33 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 15:40:39 Hail: INFO: Coerced sorted dataset
2021-09-20 15:40:41 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 15:40:48 Hail: INFO: Coerced sorted dataset
2021-09-20 16:03:06 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-20 16:03:09 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 16:03:14 Hail: INFO: Coerced sorted dataset
2021-09-20 16:24:26 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-20 16:24:31 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 16:24:33 Hail: INFO: Coerced sorted dataset
2021-09-20 16:45:53 Hail: INFO: Ordering unsorted dataset with network 

+-------------------+----------------------+---------------+----------------------+---------------+
| Gene              | '1705888'.singletons | '1705888'.pKO | '3544034'.singletons | '3544034'.pKO |
+-------------------+----------------------+---------------+----------------------+---------------+
| str               |                int64 |       float64 |                int64 |       float64 |
+-------------------+----------------------+---------------+----------------------+---------------+
| "ENSG00000008735" |                   NA |            NA |                   NA |            NA |
| "ENSG00000015475" |                   NA |            NA |                   NA |            NA |
| "ENSG00000025770" |                   NA |            NA |                   NA |            NA |
| "ENSG00000040608" |                   NA |            NA |                   NA |            NA |
| "ENSG00000054611" |                   NA |            NA |                   NA |            NA |
| "ENSG00000056487" |                   NA |            NA |                   NA |            NA |
| "ENSG00000063515" |                   NA |            NA |                   NA |            NA |
| "ENSG00000069998" |                   NA |            NA |                   NA |            NA |
| "ENSG00000070010" |                   NA |            NA |                   NA |            NA |
| "ENSG00000070371" |                   NA |            NA |                   NA |            NA |
| "ENSG00000070413" |                   NA |            NA |                   NA |            NA |
| "ENSG00000073146" |                   NA |            NA |                   NA |            NA |
| "ENSG00000073150" |                   NA |            NA |                   NA |            NA |
| "ENSG00000073169" |                   NA |            NA |                   NA |            NA |
| "ENSG00000075218" |                   NA |            NA |                   NA |            NA |
| "ENSG00000075234" |                   NA |            NA |                   NA |            NA |
| "ENSG00000075240" |                   NA |            NA |                   NA |            NA |
| "ENSG00000075275" |                   NA |            NA |                   NA |            NA |
| "ENSG00000077935" |                   NA |            NA |                   NA |            NA |
| "ENSG00000077942" |                   NA |            NA |                   NA |            NA |
| "ENSG00000079974" |                   NA |            NA |                   NA |            NA |
| "ENSG00000093000" |                   NA |            NA |                   NA |            NA |
| "ENSG00000093009" |                   NA |            NA |                   NA |            NA |
| "ENSG00000093010" |                   NA |            NA |                   NA |            NA |
| "ENSG00000093072" |                   NA |            NA |                   NA |            NA |
| "ENSG00000099889" |                   NA |            NA |                   NA |            NA |
| "ENSG00000099901" |                   NA |            NA |                   NA |            NA |
| "ENSG00000099904" |                   NA |            NA |                   NA |            NA |
| "ENSG00000099910" |                   NA |            NA |                   NA |            NA |
+-------------------+----------------------+---------------+----------------------+---------------+
showing top 29 rows

In [33]:
mt_rs = gene_csqs_rsid_builder(mt1)
mt_rs = mt_rs.filter_entries(hl.literal(set(['CH','HO'])).contains(mt_rs.csqs))
mt_rs.show()

2021-09-20 16:58:03 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 16:58:03 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 16:58:26 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 16:58:30 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 16:58:37 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 16:58:45 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 16:59:18 Hail: INFO: Coerced almost-sorted dataset


FatalError: OutOfMemoryError: GC overhead limit exceeded

Java stack trace:
java.lang.OutOfMemoryError: GC overhead limit exceeded
	at scala.collection.mutable.ResizableArray.ensureSize(ResizableArray.scala:106)
	at scala.collection.mutable.ResizableArray.ensureSize$(ResizableArray.scala:96)
	at scala.collection.mutable.ArrayBuffer.ensureSize(ArrayBuffer.scala:49)
	at scala.collection.mutable.ArrayBuffer.$plus$eq(ArrayBuffer.scala:85)
	at scala.collection.mutable.ArrayBuffer.$plus$eq(ArrayBuffer.scala:49)
	at scala.collection.generic.Growable.$anonfun$$plus$plus$eq$1(Growable.scala:62)
	at scala.collection.generic.Growable$$Lambda$8/233996206.apply(Unknown Source)
	at scala.collection.Iterator.foreach(Iterator.scala:941)
	at scala.collection.Iterator.foreach$(Iterator.scala:941)
	at scala.collection.AbstractIterator.foreach(Iterator.scala:1429)
	at scala.collection.generic.Growable.$plus$plus$eq(Growable.scala:62)
	at scala.collection.generic.Growable.$plus$plus$eq$(Growable.scala:53)


2021-09-20 17:00:51 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 17:01:58 Hail: INFO: Coerced almost-sorted dataset


FatalError: OutOfMemoryError: Java heap space

Java stack trace:
org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 100.0 failed 1 times, most recent failure: Lost task 0.0 in stage 100.0 (TID 3367) (compe018.hpc.in.bmrc.ox.ac.uk executor driver): java.lang.OutOfMemoryError: Java heap space
	at com.esotericsoftware.kryo.io.Output.require(Output.java:172)
	at com.esotericsoftware.kryo.io.Output.writeBytes(Output.java:251)
	at com.esotericsoftware.kryo.io.Output.writeBytes(Output.java:237)
	at com.esotericsoftware.kryo.serializers.DefaultArraySerializers$ByteArraySerializer.write(DefaultArraySerializers.java:49)
	at com.esotericsoftware.kryo.serializers.DefaultArraySerializers$ByteArraySerializer.write(DefaultArraySerializers.java:38)
	at com.esotericsoftware.kryo.Kryo.writeObjectOrNull(Kryo.java:629)
	at com.esotericsoftware.kryo.serializers.DefaultArraySerializers$ObjectArraySerializer.write(DefaultArraySerializers.java:332)
	at com.esotericsoftware.kryo

In [21]:
def gene_csqs_case_builder_legacy(in_mt):
    
    # get all snps that are not homozygous
    mt = in_mt
    mt = analysis.annotate_phased_entries(mt)
    mt = mt.filter_entries(~mt.GT.is_hom_var())
    # create table for each strand and combine to gene
    ht0 = (mt.group_rows_by(mt.vep.Gene).aggregate(s0 = hl.agg.any(mt.a0_alt))).entries()
    ht1 = (mt.group_rows_by(mt.vep.Gene).aggregate(s1 = hl.agg.any(mt.a1_alt))).entries()
    ht = ht0.annotate(s1 = ht1[ht0.Gene, ht0.s].s1)
    # get all snps that are homozygous
    ht2 = (in_mt.group_rows_by(in_mt.vep.Gene).aggregate(hom_var = hl.agg.any(in_mt.GT.is_hom_var()))).entries()
    ht = ht.annotate(hom_var = ht2[ht.Gene, ht.s].hom_var)
    # combine into single annotation
    expr = (hl.case()
           .when( (ht.s0) & (ht.s1) & (ht.hom_var), 'CH+HO')
           .when( (ht.s0) & (ht.s1), 'CH')
           .when( (ht.hom_var), 'HO')
           .when( (ht.s0) & (ht.s1 == False), 'HE')
           .when( (ht.s1) & (ht.s0 == False), 'HE')
           .default(''))
    ht = ht.annotate(gene_csqs = expr)
    # drop annotations not used
    return ht

def gene_csqs_case_builder(in_mt):
    r''' Returns matrix table that contains gene consequence information from phased geneotypes.
    "": no alternate alleles,
    "HE": one alternate allele on either strand in a locus, 
    "HO": homozygous for alternate alleles
    "CH": two alternate allele on either strand in a locus (compound heterozygous)
    "CH+HO": two alternate allele on either strand in a locus (either as homozygous or compound heterozygous)
    '''
    # get all snps that are not homozygous
    mt = in_mt
    mt = analysis.annotate_phased_entries(mt)
    mt = mt.filter_entries(~mt.GT.is_hom_var())
    # create table for each strand and combine to gene
    ht0 = (mt.group_rows_by(mt.vep.Gene).aggregate(s0 = hl.agg.any(mt.a0_alt)))
    ht1 = (mt.group_rows_by(mt.vep.Gene).aggregate(s1 = hl.agg.any(mt.a1_alt)))
    ht2 = (in_mt.group_rows_by(in_mt.vep.Gene).aggregate(hom_var = hl.agg.any(in_mt.GT.is_hom_var())))
    # combine entries
    ht = ht0.annotate_entries(s1 = ht1[ht0.Gene, ht0.s].s1)
    ht = ht.annotate_entries(hom_var = ht2[ht.Gene, ht.s].hom_var)
    expr = (hl.case()
           .when( (ht.s0) & (ht.s1) & (ht.hom_var), 'CH+HO')
           .when( (ht.s0) & (ht.s1), "CH")
           .when( (ht.hom_var), 'HO')
           .when( (ht.s0) & (ht.s1 == False), 'HE')
           .when( (ht.s1) & (ht.s0 == False), 'HE')
           .default(''))
    ht = ht.annotate_entries(csqs = expr)
    ht = ht.drop('s0').drop('s1').drop('hom_var')    
    return ht

def gene_csqs_dosage_builder(in_mt):
    r''' Returns matrix table that contains dosage information from phased geneotypes.
    0: two refererence alleles in locus,
    1: one alternate allele on either strand in a locus, 
    2: two alternate allele on either strand in a locus (either as homozygous or compound heterozygous)
    '''
    # get all snps that are not homozygous
    mt = in_mt
    mt = analysis.annotate_phased_entries(mt)
    mt = mt.filter_entries(~mt.GT.is_hom_var())
    # create table for each strand and combine to gene
    ht0 = (mt.group_rows_by(mt.vep.Gene).aggregate(s0 = hl.agg.any(mt.a0_alt)))
    ht1 = (mt.group_rows_by(mt.vep.Gene).aggregate(s1 = hl.agg.any(mt.a1_alt)))
    ht2 = (in_mt.group_rows_by(in_mt.vep.Gene).aggregate(hom_var = hl.agg.any(in_mt.GT.is_hom_var())))
    # combine entries
    ht = ht0.annotate_entries(s1 = ht1[ht0.Gene, ht0.s].s1)
    ht = ht.annotate_entries(hom_var = ht2[ht.Gene, ht.s].hom_var)
    expr = (hl.case()
           .when( (ht.s0) & (ht.s1) & (ht.hom_var), 2)
           .when( (ht.s0) & (ht.s1), 2)
           .when( (ht.hom_var), 2)
           .when( (ht.s0) & (ht.s1 == False), 1)
           .when( (ht.s1) & (ht.s0 == False), 1)
           .default(0))
    ht = ht.annotate_entries(DT = expr)
    ht = ht.drop('s0').drop('s1').drop('hom_var')    
    return ht



    

In [26]:
def gene_csqs_vep_builder(in_mt):
    r'''Makes a matrix table that contains variants for each (phased) strand'''
    # annotate with rsid
    in_mt = in_mt.annotate_entries(rsid_entry = in_mt.rsid)
    in_mt = in_mt.annotate_entries(rsid_gt = hl.delimit([in_mt.rsid_entry,
                                                         hl.str(in_mt.GT),
                                                         in_mt.vep.consequence_category,
                                                         in_mt.vep.most_severe_consequence,
                                                         in_mt.vep.LoF,
                                                         in_mt.vep.LoF_flags,
                                                         hl.str(in_mt.vep.revel_score),
                                                         hl.str(in_mt.vep.cadd_phred_score)], ';'))
    #in_mt = in_mt.annotate_entries(rsid_gt = hl.delimit([in_mt.rsid_entry, in_mt.vep.consequence]))
    # get all snps that are not homozygous
    mt = in_mt
    mt = analysis.annotate_phased_entries(mt)
    mt = mt.filter_entries(~mt.GT.is_hom_var())
    # create table for each strand and combine to gene
    ht0 = (mt.group_rows_by(mt.vep.Gene).aggregate(s0 = hl.agg.filter(mt.a0_alt, hl.agg.collect(mt.rsid_gt))))
    ht1 = (mt.group_rows_by(mt.vep.Gene).aggregate(s1 = hl.agg.filter(mt.a1_alt, hl.agg.collect(mt.rsid_gt))))
    ht2 = (in_mt.group_rows_by(in_mt.vep.Gene).aggregate(hom_var = hl.agg.filter(in_mt.GT.is_hom_var(), hl.agg.collect(in_mt.rsid_gt))))
    # combine entries
    ht = ht0.annotate_entries(s1 = ht1[ht0.Gene, ht0.s].s1)
    ht = ht.annotate_entries(hom_var = ht2[ht.Gene, ht.s].hom_var)
    return ht


def gene_csqs_rsid_builder(in_mt, keep = ['CH','HO','CH+HO']):
    r'''Makes a matrix that cotanins knockout type and the variant (rsID) involved in the KO'''
    # build gene x variants/ko status
    mt_rs = gene_csqs_vep_builder(mt1)
    mt_dt = gene_csqs_case_builder(mt1)
    # combine 
    combined = mt_rs.annotate_entries(csqs = mt_dt[mt_rs.Gene, mt_rs.s].csqs)
    combined = combined.filter_entries(hl.literal(keep).contains(combined.csqs))
    return combined

In [29]:
gene_csqs_vep_builder(mt1).describe()
    #combined.entries().show()
#mt1.describe()
#res = gene_csqs_dosage_builder(mt1)
#res.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'Gene': str
----------------------------------------
Entry fields:
    's0': array<str>
    's1': array<str>
    'hom_var': array<str>
----------------------------------------
Column key: ['s']
Row key: ['Gene']
----------------------------------------


In [78]:
res = gene_csqs_dosage_builder(mt1)
res.show()

2021-09-17 16:15:24 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:15:33 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:15:36 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:15:40 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:15:42 Hail: INFO: Coerced sorted dataset
2021-09-17 16:17:22 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-17 16:17:22 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:17:23 Hail: INFO: Coerced sorted dataset
2021-09-17 16:19:04 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-17 16:19:05 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:19:06 Hail: INFO: Coerced sorted dataset
2021-09-17 16:20:46 Hail: INFO: Ordering unsorted dataset with network shuffle


,,
,'1705888','3544034'
Gene,DT,DT
str,int32,int32
"""ENSG00000008735""",0,0
"""ENSG00000015475""",0,0
"""ENSG00000025770""",0,0
"""ENSG00000040608""",0,0
"""ENSG00000054611""",0,0
"""ENSG00000056487""",0,0
"""ENSG00000063515""",0,0


In [43]:
res.filter(hl.literal(['CH','HO']).contains(res.gene_csqs)).show()

2021-09-17 15:53:59 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 15:54:01 Hail: INFO: Coerced sorted dataset
2021-09-17 15:55:51 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-17 15:55:51 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 15:55:53 Hail: INFO: Coerced sorted dataset
2021-09-17 15:57:39 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-17 15:57:40 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 15:57:42 Hail: INFO: Coerced sorted dataset
2021-09-17 15:59:27 Hail: INFO: Ordering unsorted dataset with network shuffle


,,,,,
Gene,s,s0,s1,hom_var,gene_csqs
str,str,bool,bool,bool,str
"""ENSG00000133454""","""1705888""",True,True,False,"""CH"""
"""ENSG00000184113""","""1705888""",False,False,True,"""HO"""
"""ENSG00000215568""","""3544034""",False,False,True,"""HO"""


In [17]:
res.show()

2021-09-17 15:36:30 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 15:36:32 Hail: INFO: Coerced sorted dataset
2021-09-17 15:38:17 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-17 15:38:18 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 15:38:19 Hail: INFO: Coerced sorted dataset
2021-09-17 15:40:04 Hail: INFO: Ordering unsorted dataset with network shuffle


,,,
Gene,s,s0,s1
str,str,bool,bool
"""ENSG00000133454""","""1705888""",True,True


In [76]:
ht0_vars.show()

2021-09-17 16:12:06 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:12:08 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:12:10 Hail: INFO: Coerced sorted dataset
2021-09-17 16:13:50 Hail: INFO: Ordering unsorted dataset with network shuffle


,,
,'1705888','3544034'
Gene,rsid,rsid
str,array<str>,array<str>
"""ENSG00000008735""",[],[]
"""ENSG00000015475""",[],[]
"""ENSG00000025770""",[],[]
"""ENSG00000040608""",[],[]
"""ENSG00000054611""",[],[]
"""ENSG00000056487""",[],[]
"""ENSG00000063515""",[],[]


In [ ]:
def calc_p_ko(mt):
    '''Annotates entries with P(Knockout). Requires, that fields are
       already annotated with "singletons" count, and "DT". '''
    ko_mt = mt.annotate_entries(
        pKO = hl.if_else(
            mt.DT == 2, 1, # knockout
            hl.if_else(
                mt.DT == 1, 
                hl.if_else(mt.singletons >= 1, 1 - (1/2)**mt.singletons, 0), # one phased hetz
                hl.if_else(mt.singletons >= 2, 1 - 2*(1/2)**mt.singletons, 0), # zero phased hetz
            )
        )
    )
    return ko_mt

In [ ]:
def gene_csqs_calc_pKO(mt_phased, mt_unphased, fields_drop = ['dosage','sigletons']):
    
    '''
    Annotates entries with P(Knockout). Requires a phased matrix and an unphased
    matrix that only contains singletons.
    '''
    
    # setup variables
    mt1 = mt_phased # contains phased non-singletons
    mt2 = mt_unphased # contains unphased singletons
    
    # Determine probability of being KO given singletons and phased hetz
    mt1_burden = gene_csqs_dosage_builder(mt1)
    mt2_burden = gene_burden_annotations_per_sample(mt2)
    mt_ko = mt1_burden.annotate_entries(singletons = mt2_burden[(mt1_burden.Gene, mt1_burden.s)].n)
    mt_ko = mt_ko.annotate_entries(singletons = hl.if_else(~hl.is_missing(mt_ko.singletons), mt_ko.singletons, 0 ))
    mt_ko = calc_p_ko(mt_ko)

    # drop not needed rows
    mt_ko = mt_ko.drop(*[f for f in fields_drop if f in mt_ko.entry])
    #mt_ko_entries = mt_ko.entries()
    #mt_ko_entries = mt_ko_entries.filter(~hl.is_missing(mt_ko_entries.pKO))
    return mt_ko

In [ ]:
def gene_csqs_calc_pKO_pseudoSNP(mt1, mt2, chrom):
    '''Calculate probability of being a knockout incoporating phased 
       data (mt1) and unphased singletons (mt2). Create a file with 
       fake markers, that can be inputted into SAIGE'''
    # mt1 is phased
    # mt2 is unphased

    # get probability matrix
    pmt = get_prob_ko_matrix(mt1, mt2, ["DT","singletons"])
    pmt = pmt.annotate_entries(DT = pmt.pKO*2) # multiply probability by 2 (dosage encoded [0:2])
    pmt = pmt.drop('pKO')

    # create fake loci
    pmt = pmt.annotate_rows(locus = hl.parse_locus('chr' + str(chrom) + ':1'))
    pmt = pmt.annotate_rows(alleles = hl.literal(['X','Y']))
    pmt = pmt.annotate_rows(rsid = pmt.Gene)
    pmt = pmt.key_rows_by(pmt.locus, pmt.alleles)
    pmt = pmt.drop('Gene')
    return pmt

In [79]:
#get_prob_ko_matrix(mt1, mt2)
analysis.construct_phased_dosage_mt(mt1).show()

2021-09-17 16:21:40 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:21:42 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:21:43 Hail: INFO: Coerced sorted dataset
2021-09-17 16:23:21 Hail: INFO: Ordering unsorted dataset with network shuffle


,,
,'1705888','3544034'
Gene,dosage,dosage
str,int32,int32
"""ENSG00000008735""",0,0
"""ENSG00000015475""",0,0
"""ENSG00000025770""",0,0
"""ENSG00000040608""",0,0
"""ENSG00000054611""",0,0
"""ENSG00000056487""",0,0
"""ENSG00000063515""",0,0


In [80]:
analysis.construct_phased_dosage_mt(mt1).describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'Gene': str
----------------------------------------
Entry fields:
    'dosage': int32
----------------------------------------
Column key: ['s']
Row key: ['Gene']
----------------------------------------


In [97]:
def is_phased(mt):
    ''' Check if the input contains phased data. Returns Bool'''
    mt = mt.annotate_entries(phased = 
                        (mt.GT ==  hl.parse_call('0|0')) |
                        (mt.GT ==  hl.parse_call('1|0')) |
                        (mt.GT ==  hl.parse_call('0|1')) |
                        (mt.GT ==  hl.parse_call('1|1'))
                       )
    return mt.aggregate_entries(hl.agg.any(mt.phased))



In [98]:
is_phased(mt1)

2021-09-17 16:46:27 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:46:29 Hail: INFO: Coerced sorted dataset


True

In [105]:
def maf_category_case_builder(mt):
    return (hl.case()
            .when(call_stats_expr.AF <= 0.00001, 0.00001)
            .when(call_stats_expr.AF <= 0.0001, 0.0001)
            .when(call_stats_expr.AF <= 0.001, 0.001)
            .when(call_stats_expr.AF <= 0.01, 0.01)
            .when(call_stats_expr.AF <= 0.1, 0.1)
            .default(0.99))

def mac_category_case_builder(call_stats_expr):
    return (hl.case()
            .when(call_stats_expr.AC <= 5, call_stats_expr.AC)
            .when(call_stats_expr.AC <= 10, 10)
            .when(call_stats_expr.AC <= 25, 25)
            .when(call_stats_expr.AC <= 100, 100)
            .when(call_stats_expr.AC <= 1000, 1000)
            .when(call_stats_expr.AC <= 10000, 10000)
            .when(call_stats_expr.AC <= 100000, 100000)
            .when(call_stats_expr.AC <= 1000000, 1000000)
            .default(0))

In [107]:
mac_category_case_builder(mt.info).show()

2021-09-17 17:28:26 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 17:28:27 Hail: INFO: Coerced sorted dataset


,,
locus,alleles,
locus<GRCh38>,array<str>,int32
chr22:15528363,"[""C"",""T""]",0
chr22:15528649,"[""G"",""A""]",0
chr22:15528650,"[""G"",""A""]",0
chr22:15528668,"[""G"",""A""]",0
chr22:15528753,"[""C"",""T""]",0
chr22:16591038,"[""G"",""A""]",0
chr22:16591083,"[""G"",""A""]",0
chr22:16591104,"[""C"",""A""]",0
